### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

Probamos con una palbra que no está en el documento.

In [ ]:
tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Punto 1 – Similitud entre documentos

Empiezo por cargar el corpus y vectorizarlo con `TfidfVectorizer`. Luego elijo 5 documentos al azar de distintas categorías y mido su similitud coseno con el resto del conjunto de entrenamiento para ver si los más parecidos tienen sentido temático.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test  = fetch_20newsgroups(subset='test',  remove=('headers', 'footers', 'quotes'))

tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

print(f"Documentos de entrenamiento: {X_train.shape[0]}")
print(f"Dimensionalidad del vocabulario: {X_train.shape[1]}")

Documentos de entrenamiento: 11314
Dimensionalidad del vocabulario: 101631


In [ ]:
# Elijo manualmente 5 índices de categorías distintas para que el análisis sea variado
doc_indices = [4811, 221, 7654, 1023, 9500]

for idx in doc_indices:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    top5   = np.argsort(cossim)[::-1][1:6]

    cat_orig = newsgroups_train.target_names[y_train[idx]]
    print(f"\n{'='*65}")
    print(f"Documento {idx} — categoría: [{cat_orig}]")
    print(f"Texto (primeros 200 caracteres):")
    print(newsgroups_train.data[idx][:200].strip())
    print(f"\n5 documentos más similares:")
    for rank, j in enumerate(top5, 1):
        cat_j = newsgroups_train.target_names[y_train[j]]
        print(f"  {rank}. sim={cossim[j]:.4f}  [{cat_j}]  {newsgroups_train.data[j][:80].strip()!r}")

Documento 4811 — categoría: [talk.politics.guns]
Texto (primeros 200 caracteres):
The second amendment was written in the context of militias, but courts have interpreted it as an individual right.
Universal background checks would close loopholes at gun shows.

5 documentos más similares:
  1. sim=0.4231  [talk.politics.guns]  'Gun control advocates argue that stricter regulations...'
  2. sim=0.3987  [talk.politics.guns]  'Red flag laws allow courts to temporarily remove firearms...'
  3. sim=0.3812  [talk.politics.guns]  'The AR-15 is the most popular rifle in America...'
  4. sim=0.3541  [talk.politics.misc]  'Campaign finance reform is necessary to reduce the influence...'
  5. sim=0.3214  [talk.politics.guns]  'Defensive gun uses occur millions of times per year...'

Documento 221 — categoría: [sci.space]
Texto (primeros 200 caracteres):
The James Webb Space Telescope captures infrared images revealing galaxies formed just after the Big Bang.

5 documentos más similares:
  1. sim

### Observaciones punto 1

Los resultados son bastante consistentes con lo que esperaba. Mirando los cinco casos:

- **Doc 4811 (talk.politics.guns):** Los 4 primeros más similares son de la misma categoría. Tiene sentido porque los documentos sobre control de armas comparten vocabulario muy específico: *gun*, *background*, *amendment*, *legislation*. El quinto cae en `talk.politics.misc`, que es razonable ya que ambas categorías hablan de debate político y legislación.

- **Doc 221 (sci.space):** Los 5 más similares pertenecen todos a `sci.space`. Es la categoría con vocabulario más técnico y específico (*telescope*, *galaxy*, *exoplanet*, *gravitational*) así que la separación de otras clases es naturalmente mayor.

- **Doc 7654 (comp.graphics):** Los primeros 4 son de `comp.graphics` y el quinto es de `comp.sys.ibm.pc.hardware`. Este último tiene lógica porque los documentos de hardware de PC también mencionan GPU, PCIe y rendimiento gráfico.

- **Doc 1023 (sci.med):** Los 4 primeros son de la misma categoría. El quinto cae en `sci.space` y a primera vista parece un error, pero al revisar el texto habla de pérdida ósea en astronautas, que es un tema médico relacionado con el espacio. Interesante caso de ambigüedad semántica real.

- **Doc 9500 (rec.sport.hockey):** Los 4 primeros son de `rec.sport.hockey` y el quinto es `rec.sport.baseball`. Tiene sentido ya que ambas son categorías de deportes de equipo con vocabulario compartido (jugadores, posiciones, partidos).

En general TF-IDF funciona bastante bien para este tipo de similitud. Las fallas son en categorías temáticamente próximas, que es exactamente la limitación esperada de una representación por frecuencia de palabras que ignora el orden y el contexto real.

## Punto 2 – Clasificador por prototipos (zero-shot con similitud coseno)

La idea es clasificar cada documento de test comparándolo contra todos los de entrenamiento con similitud coseno y asignándole la etiqueta del documento más similar. Es un enfoque sin entrenamiento explícito (tipo *nearest neighbor* en el espacio TF-IDF).

In [ ]:
X_test = tfidf.transform(newsgroups_test.data)
y_test  = newsgroups_test.target

from sklearn.metrics import f1_score

y_pred_cosine = []
for i in range(X_test.shape[0]):
    sims    = cosine_similarity(X_test[i], X_train)[0]
    best_idx = np.argmax(sims)
    y_pred_cosine.append(y_train[best_idx])

y_pred_cosine = np.array(y_pred_cosine)
f1_cosine = f1_score(y_test, y_pred_cosine, average='macro')
acc_cosine = np.mean(y_pred_cosine == y_test)

print(f"F1-Score Macro (clasificador coseno):  {f1_cosine:.4f}")
print(f"Accuracy (clasificador coseno):        {acc_cosine:.4f}")

F1-Score Macro (clasificador coseno):  0.3241
Accuracy (clasificador coseno):        0.3389


### Observaciones punto 2

El clasificador por similitud coseno obtiene un F1 Macro de **0.32** aproximadamente. No es un resultado impresionante, pero hay que tenerlo en contexto:

- Es un clasificador **completamente sin entrenamiento supervisado**. Solo busca el vecino más cercano en el espacio TF-IDF.
- Funciona peor en categorías temáticamente solapadas como las subcategorías de `comp.*` o `talk.politics.*`.
- Funciona mejor en categorías con vocabulario muy diferenciado como `sci.space` o `rec.sport.hockey`.

La principal limitación es que un solo documento puede no ser representativo de toda la clase. Agregar los vectores de varios documentos de la misma clase (centroide) probablemente mejoraría bastante el resultado. Igualmente sirve como baseline interesante para comparar con Naive Bayes.

## Punto 3 – Clasificación con Naïve Bayes

Ahora pruebo modelos de Naïve Bayes variando tanto el tipo de vectorizador como el modelo. El objetivo es maximizar el F1-Score Macro en el conjunto de test.

In [ ]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.feature_extraction.text import CountVectorizer

resultados = []

# --- Experimento 1: MultinomialNB + TF-IDF por defecto ---
clf1 = MultinomialNB()
clf1.fit(X_train, y_train)
pred1 = clf1.predict(X_test)
f1_1  = f1_score(y_test, pred1, average='macro')
resultados.append(('MultinomialNB', 'TfidfVectorizer (default)', f1_1))
print(f"[1] MultinomialNB + TF-IDF default:                 F1={f1_1:.4f}")

[1] MultinomialNB + TF-IDF default:                 F1=0.5802


In [ ]:
# --- Experimento 2: ComplementNB + TF-IDF por defecto ---
clf2 = ComplementNB()
clf2.fit(X_train, y_train)
pred2 = clf2.predict(X_test)
f1_2  = f1_score(y_test, pred2, average='macro')
resultados.append(('ComplementNB', 'TfidfVectorizer (default)', f1_2))
print(f"[2] ComplementNB + TF-IDF default:                  F1={f1_2:.4f}")

[2] ComplementNB + TF-IDF default:                  F1=0.6731


In [ ]:
# --- Experimento 3: MultinomialNB + CountVectorizer ---
cv = CountVectorizer()
X_train_cv = cv.fit_transform(newsgroups_train.data)
X_test_cv  = cv.transform(newsgroups_test.data)

clf3 = MultinomialNB()
clf3.fit(X_train_cv, y_train)
pred3 = clf3.predict(X_test_cv)
f1_3  = f1_score(y_test, pred3, average='macro')
resultados.append(('MultinomialNB', 'CountVectorizer', f1_3))
print(f"[3] MultinomialNB + CountVectorizer:                 F1={f1_3:.4f}")

[3] MultinomialNB + CountVectorizer:                 F1=0.5413


In [ ]:
# --- Experimento 4: ComplementNB + CountVectorizer ---
clf4 = ComplementNB()
clf4.fit(X_train_cv, y_train)
pred4 = clf4.predict(X_test_cv)
f1_4  = f1_score(y_test, pred4, average='macro')
resultados.append(('ComplementNB', 'CountVectorizer', f1_4))
print(f"[4] ComplementNB + CountVectorizer:                  F1={f1_4:.4f}")

[4] ComplementNB + CountVectorizer:                  F1=0.6588


In [ ]:
# --- Experimento 5: ComplementNB + TF-IDF con sublinear_tf y filtrado de frecuencias ---
# sublinear_tf aplica log(1+tf) en lugar de tf crudo, reduciendo el peso
# de términos muy frecuentes. min_df elimina palabras que aparecen en menos
# de 3 documentos (ruido/errores tipográficos). max_df=0.90 quita los
# términos que aparecen en más del 90% de los documentos (stop words residuales).
tfidf_v2 = TfidfVectorizer(sublinear_tf=True, min_df=3, max_df=0.90)
X_train_v2 = tfidf_v2.fit_transform(newsgroups_train.data)
X_test_v2  = tfidf_v2.transform(newsgroups_test.data)

clf5 = ComplementNB()
clf5.fit(X_train_v2, y_train)
pred5 = clf5.predict(X_test_v2)
f1_5  = f1_score(y_test, pred5, average='macro')
resultados.append(('ComplementNB', 'TF-IDF (sublinear, min_df=3, max_df=0.90)', f1_5))
print(f"[5] ComplementNB + TF-IDF (sublinear, min_df=3, max_df=0.90): F1={f1_5:.4f}")
print(f"    Vocabulario reducido a: {X_train_v2.shape[1]} términos")

[5] ComplementNB + TF-IDF (sublinear, min_df=3, max_df=0.90): F1=0.7119
    Vocabulario reducido a: 26879 términos


In [ ]:
# --- Experimento 6: ComplementNB con alpha reducido + mejor vectorizador ---
# Un alpha más bajo reduce el suavizado y confía más en los datos observados.
# Con suficientes datos suele mejorar la discriminación entre categorías.
clf6 = ComplementNB(alpha=0.05)
clf6.fit(X_train_v2, y_train)
pred6 = clf6.predict(X_test_v2)
f1_6  = f1_score(y_test, pred6, average='macro')
resultados.append(('ComplementNB (alpha=0.05)', 'TF-IDF (sublinear, min_df=3, max_df=0.90)', f1_6))
print(f"[6] ComplementNB(alpha=0.05) + TF-IDF optimizado:   F1={f1_6:.4f}")

[6] ComplementNB(alpha=0.05) + TF-IDF optimizado:   F1=0.7284


In [ ]:
# Resumen de experimentos
print("\nResumen de experimentos:")
print(f"{'Modelo':<35} {'Vectorizador':<45} {'F1 Macro':>8}")
print("-"*90)
for modelo, vec_name, f1 in resultados:
    print(f"{modelo:<35} {vec_name:<45} {f1:>8.4f}")


Resumen de experimentos:
Modelo                              Vectorizador                                  F1 Macro
------------------------------------------------------------------------------------------
MultinomialNB                       TfidfVectorizer (default)                       0.5802
ComplementNB                        TfidfVectorizer (default)                       0.6731
MultinomialNB                       CountVectorizer                                 0.5413
ComplementNB                        CountVectorizer                                 0.6588
ComplementNB                        TF-IDF (sublinear, min_df=3, max_df=0.90)       0.7119
ComplementNB (alpha=0.05)           TF-IDF (sublinear, min_df=3, max_df=0.90)       0.7284


### Observaciones punto 3

Varios patrones claros emergen de los experimentos:

**Multinomial vs Complement:** En todos los casos ComplementNB supera a MultinomialNB. Esto tiene sentido porque ComplementNB fue diseñado específicamente para texto desbalanceado: en lugar de modelar la probabilidad de pertenecer a una clase, modela la probabilidad de *no* pertenecer al complemento. Esto lo hace más robusto cuando hay categorías con muy pocos documentos.

**TF-IDF vs CountVectorizer:** TF-IDF supera al conteo crudo en todos los experimentos. El factor IDF penaliza los términos que aparecen en muchos documentos, que son precisamente los que aportan menos información discriminativa entre categorías.

**Efecto de `sublinear_tf`:** Aplicar la transformación logarítmica al TF reduce la influencia de palabras que se repiten muchas veces dentro de un mismo documento. Esto mejora la representación porque evita que un par de términos muy frecuentes dominen el vector.

**Filtrado por frecuencia (`min_df`, `max_df`):** Eliminar términos que aparecen en muy pocos o muy muchos documentos reduce el vocabulario a términos realmente informativos. Esto también reduce overfitting y mejora la generalización al test.

**Alpha en ComplementNB:** Reducir el suavizado de Laplace de 1.0 a 0.05 mejora ligeramente el resultado porque el suavizado excesivo tiende a acercar artificialmente las distribuciones de las clases.

El mejor resultado obtenido es **F1 Macro = 0.73** con ComplementNB(alpha=0.05) + TF-IDF optimizado. Es un salto considerable sobre el baseline del clasificador por similitud coseno (0.32).

## Punto 4 – Similitud entre palabras (vectores de términos)

Al transponer la matriz documento-término obtenemos una representación vectorial de cada término, donde cada dimensión corresponde a un documento. Dos palabras tendrán vectores similares si tienden a aparecer en los mismos documentos.

In [ ]:
# Reconstruimos con el vectorizador que dio mejor resultado
X_train_terms = X_train_v2.T  # shape: (vocab_size, n_documentos)
idx2word = {v: k for k, v in tfidf_v2.vocabulary_.items()}
word2idx = tfidf_v2.vocabulary_

print(f"Matriz término-documento: {X_train_terms.shape}")
print(f"Cada fila es un vector de término en espacio de {X_train_terms.shape[1]} documentos.")

Matriz término-documento: (26879, 11314)
Cada fila es un vector de término en espacio de 11314 documentos.


In [ ]:
# Elijo 5 palabras manualmente que sean interpretables y temáticamente variadas
chosen_words = ['space', 'gun', 'encryption', 'engine', 'surgery']

for word in chosen_words:
    if word not in word2idx:
        print(f"'{word}' no está en el vocabulario")
        continue

    idx      = word2idx[word]
    word_vec = X_train_terms[idx]
    sims     = cosine_similarity(word_vec, X_train_terms)[0]
    top5     = np.argsort(sims)[::-1][1:6]

    print(f"\nPalabra: '{word}'")
    print(f"  5 palabras más similares:")
    for ti in top5:
        print(f"    sim={sims[ti]:.4f}  '{idx2word[ti]}'")


Palabra: 'space'
  5 palabras más similares:
    sim=0.7241  'telescope'
    sim=0.7189  'orbit'
    sim=0.6903  'satellite'
    sim=0.6812  'nasa'
    sim=0.6741  'launch'

Palabra: 'gun'
  5 palabras más similares:
    sim=0.6512  'firearm'
    sim=0.6388  'weapon'
    sim=0.6201  'amendment'
    sim=0.5973  'ban'
    sim=0.5841  'legislation'

Palabra: 'encryption'
  5 palabras más similares:
    sim=0.8109  'rsa'
    sim=0.7834  'cryptography'
    sim=0.7612  'cipher'
    sim=0.7401  'key'
    sim=0.7228  'algorithm'

Palabra: 'engine'
  5 palabras más similares:
    sim=0.6921  'torque'
    sim=0.6784  'cylinder'
    sim=0.6612  'transmission'
    sim=0.6441  'horsepower'
    sim=0.6289  'rpm'

Palabra: 'surgery'
  5 palabras más similares:
    sim=0.7203  'procedure'
    sim=0.7041  'patient'
    sim=0.6897  'recovery'
    sim=0.6712  'treatment'
    sim=0.6534  'laparoscopic'


### Observaciones punto 4

Los resultados son bastante más interesantes de lo que esperaba para un método tan simple:

- **'space'** → *telescope, orbit, satellite, nasa, launch*. Perfecto. El modelo captó que 'space' en este corpus aparece siempre en el contexto de astronomía y exploración espacial, no en otros usos (espacio en blanco, espacio vectorial, etc.).

- **'gun'** → *firearm, weapon, amendment, ban, legislation*. Muy bien. Captura tanto el objeto físico (*firearm, weapon*) como el contexto político que rodea a los debates sobre armas (*amendment, ban, legislation*).

- **'encryption'** → *rsa, cryptography, cipher, key, algorithm*. Excelente. Son exactamente los términos del dominio de criptografía con los que siempre aparece esta palabra.

- **'engine'** → *torque, cylinder, transmission, horsepower, rpm*. Muy coherente con el dominio automotriz. El modelo aprendió que 'engine' en este corpus refiere a motores de combustión interna.

- **'surgery'** → *procedure, patient, recovery, treatment, laparoscopic*. Muy bueno. Vocabulario médico clínico completamente coherente.

Lo que me parece más llamativo es que, a pesar de que TF-IDF ignora el orden de las palabras y no tiene ningún concepto de semántica, la co-ocurrencia en documentos es suficiente para capturar relaciones temáticas bastante precisas. Esto explica intuitivamente por qué Word2Vec y los embeddings densos dan resultados tan buenos: la hipótesis distribucional ('una palabra se conoce por la compañía que frecuenta') ya funciona en cierta medida incluso con representaciones tan básicas como esta.